# DEIMv2-Atto — Pill Fine-tuning
**Runtime → Change runtime type → T4 GPU**  
Estimated time: ~2–4 hours on T4 for 72 epochs.

## 1. Install dependencies

In [ ]:
!pip install torch==2.5.1 torchvision==0.20.1 -q
!pip install faster-coco-eval PyYAML scipy calflops transformers safetensors roboflow onnxruntime onnx -q
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## 2. Clone DEIMv2

In [ ]:
!git clone https://github.com/Intellindust-AI-Lab/DEIMv2
%cd DEIMv2

## 3. Download pill dataset from Roboflow
Paste your API key below. Get it at https://app.roboflow.com → Settings → API Keys

In [ ]:
ROBOFLOW_API_KEY = "PASTE_YOUR_KEY_HERE"  # <-- edit this

from roboflow import Roboflow
import json, os, shutil

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Try these in order until one works for you:
# Option A — general pill counting dataset
project = rf.workspace("roboflow-universe-projects").project("pill-counting")
# Option B — pills detection
# project = rf.workspace("seblful").project("pills-detection-s9ywn")

dataset = project.version(1).download("coco", location="./rf_download")
print('Downloaded to:', dataset.location)

In [ ]:
# Reorganise into DEIMv2's expected layout
import json, os, shutil

os.makedirs('data/pills/train2017',     exist_ok=True)
os.makedirs('data/pills/val2017',       exist_ok=True)
os.makedirs('data/pills/annotations',   exist_ok=True)

for rf_split, coco_split in [('train', 'train2017'), ('valid', 'val2017')]:
    src = os.path.join(dataset.location, rf_split)
    dst = f'data/pills/{coco_split}'
    ann_src = os.path.join(src, '_annotations.coco.json')
    ann_dst = f'data/pills/annotations/instances_{coco_split}.json'

    count = 0
    for f in os.listdir(src):
        if f.lower().endswith(('.jpg','.jpeg','.png')):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
            count += 1
    print(f'{rf_split}: {count} images')

    if os.path.isfile(ann_src):
        shutil.copy2(ann_src, ann_dst)
        with open(ann_dst) as f:
            ann = json.load(f)
        print(f'  categories: {[c["name"] for c in ann["categories"]]}')
        print(f'  annotations: {len(ann["annotations"])}')

## 4. Write the pill fine-tuning config

In [ ]:
# Count classes from the downloaded dataset
import json
with open('data/pills/annotations/instances_train2017.json') as f:
    ann = json.load(f)
num_classes = len(ann['categories'])
print(f'num_classes = {num_classes}')
print('categories:', ann['categories'])

In [ ]:
import json
with open('data/pills/annotations/instances_train2017.json') as f:
    ann = json.load(f)
num_classes = len(ann['categories'])

config_content = f"""__include__: [
  '../runtime.yml',
  '../base/dataloader.yml',
  '../base/optimizer.yml',
  '../base/deimv2.yml',
]

output_dir: ./outputs/deimv2_atto_pills

task: detection
num_classes: {num_classes}
remap_mscoco_category: False

evaluator:
  type: CocoEvaluator
  iou_types: ['bbox']

train_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: ./data/pills/train2017/
    ann_file: ./data/pills/annotations/instances_train2017.json
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: True
  num_workers: 2
  drop_last: True
  total_batch_size: 32
  collate_fn:
    type: BatchImageCollateFunction

val_dataloader:
  type: DataLoader
  dataset:
    type: CocoDetection
    img_folder: ./data/pills/val2017/
    ann_file: ./data/pills/annotations/instances_val2017.json
    return_masks: False
    transforms:
      type: Compose
      ops: ~
  shuffle: False
  num_workers: 2
  drop_last: False
  total_batch_size: 32
  collate_fn:
    type: BatchImageCollateFunction

DEIM:
  encoder: LiteEncoder

HGNetv2:
  name: 'Atto'
  return_idx: [2]
  freeze_at: -1
  freeze_norm: False
  use_lab: True

LiteEncoder:
  in_channels: [256]
  feat_strides: [16]
  hidden_dim: 64
  expansion: 0.34
  depth_mult: 0.5
  act: 'silu'

DEIMTransformer:
  feat_channels: [64, 64]
  feat_strides: [16, 32]
  hidden_dim: 64
  num_levels: 2
  num_points: [4, 2]
  num_layers: 3
  eval_idx: -1
  num_queries: 100
  dim_feedforward: 160
  share_bbox_head: True
  use_gateway: False

epoches: 72
warmup_iter: 500
flat_epoch: 36
no_aug_epoch: 8
lr_gamma: 0.5

optimizer:
  type: AdamW
  params:
    - params: '^(?=.*backbone)(?!.*norm|bn).*$'
      lr: 0.0001
    - params: '^(?=.*backbone)(?=.*norm|bn).*$'
      lr: 0.0001
      weight_decay: 0.
    - params: '^(?=.*(?:encoder|decoder))(?=.*(?:norm|bn)).*$'
      weight_decay: 0.
  lr: 0.0002
  betas: [0.9, 0.999]
  weight_decay: 0.0001

eval_spatial_size: [320, 320]

train_dataloader:
  total_batch_size: 32
  dataset:
    transforms:
      ops:
        - {{type: Mosaic, output_size: 160, rotation_range: 10, translation_range: [0.1, 0.1], scaling_range: [0.5, 1.5],
           probability: 1.0, fill_value: 0, use_cache: True, max_cached_images: 50, random_pop: True}}
        - {{type: RandomPhotometricDistort, p: 0.5}}
        - {{type: RandomZoomOut, fill: 0}}
        - {{type: RandomIoUCrop, p: 0.8}}
        - {{type: SanitizeBoundingBoxes, min_size: 8}}
        - {{type: RandomHorizontalFlip}}
        - {{type: Resize, size: [320, 320]}}
        - {{type: SanitizeBoundingBoxes, min_size: 8}}
        - {{type: ConvertPILImage, dtype: 'float32', scale: True}}
        - {{type: ConvertBoxes, fmt: 'cxcywh', normalize: True}}
      policy:
        epoch: [4, 36, 60]
      mosaic_prob: 0.3
  collate_fn:
    mixup_prob: 0.0
    mixup_epochs: [40000, 15000]
    copyblend_prob: 0.0
    copyblend_epochs: [40000, 15000]
    stop_epoch: 64
    ema_restart_decay: 0.9999
    base_size: 320
    base_size_repeat: ~

val_dataloader:
  total_batch_size: 32
  dataset:
    transforms:
      ops:
        - {{type: Resize, size: [320, 320]}}
        - {{type: ConvertPILImage, dtype: 'float32', scale: True}}
  shuffle: False
  num_workers: 2

DEIMCriterion:
  losses: ['mal', 'boxes']
  use_uni_set: False
  matcher:
    matcher_change_epoch: 64
"""

os.makedirs('configs/deimv2', exist_ok=True)
with open('configs/deimv2/deimv2_hgnetv2_atto_pills.yml', 'w') as f:
    f.write(config_content)
print('Config written with', num_classes, 'classes')

## 5. Download pretrained Atto weights

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import torch, os

os.makedirs('weights', exist_ok=True)

sf_path = hf_hub_download(
    repo_id='Intellindust/DEIMv2_HGNetv2_ATTO_COCO',
    filename='model.safetensors',
    local_dir='./weights'
)
state = load_file(sf_path)
checkpoint = {'model': state, 'ema': {'module': state}}
torch.save(checkpoint, 'weights/deimv2_atto_coco.pth')
print('Weights ready:', os.path.getsize('weights/deimv2_atto_coco.pth') // 1024, 'KB')

## 6. Fine-tune (Step 6)
~2–4 hours on T4. Loss and mAP logged to `outputs/deimv2_atto_pills/`.

In [ ]:
!python train.py \
  -c configs/deimv2/deimv2_hgnetv2_atto_pills.yml \
  --use-amp \
  --seed=42 \
  -t weights/deimv2_atto_coco.pth \
  2>&1 | tee outputs/train_log.txt

## 7. Evaluate (Step 7)

In [ ]:
# Find best checkpoint
import glob, os
checkpoints = glob.glob('outputs/deimv2_atto_pills/*.pth')
best = sorted(checkpoints)[-1]  # last = best (DEIMv2 saves best.pth)
# prefer best.pth if it exists
if os.path.isfile('outputs/deimv2_atto_pills/best.pth'):
    best = 'outputs/deimv2_atto_pills/best.pth'
print('Using checkpoint:', best)

In [ ]:
!python train.py \
  -c configs/deimv2/deimv2_hgnetv2_atto_pills.yml \
  --test-only \
  -r {best}

## 8. Export to ONNX + int8 quantize (Step 8)

In [ ]:
import glob, os
best = 'outputs/deimv2_atto_pills/best.pth'
if not os.path.isfile(best):
    checkpoints = sorted(glob.glob('outputs/deimv2_atto_pills/*.pth'))
    best = checkpoints[-1]
print('Exporting:', best)

!python tools/deployment/export_onnx.py \
  --check \
  -c configs/deimv2/deimv2_hgnetv2_atto_pills.yml \
  -r {best}

In [ ]:
# int8 dynamic quantization — cuts size ~4x with minimal accuracy loss
from onnxruntime.quantization import quantize_dynamic, QuantType
import os, glob

onnx_files = glob.glob('*.onnx') + glob.glob('outputs/**/*.onnx', recursive=True)
src = onnx_files[0]
dst = src.replace('.onnx', '_int8.onnx')

quantize_dynamic(src, dst, weight_type=QuantType.QUInt8)

fp32_kb = os.path.getsize(src) // 1024
int8_kb = os.path.getsize(dst) // 1024
print(f'fp32: {fp32_kb} KB → int8: {int8_kb} KB  (reduction: {fp32_kb/int8_kb:.1f}x)')
print('int8 model:', dst)

In [ ]:
# Download the int8 model to your machine
from google.colab import files
import glob
int8_files = glob.glob('*_int8.onnx') + glob.glob('outputs/**/*_int8.onnx', recursive=True)
if int8_files:
    files.download(int8_files[0])
    print('Downloading:', int8_files[0])
else:
    # Fallback: download fp32
    fp32_files = glob.glob('*.onnx')
    files.download(fp32_files[0])